# 02 — Pré-cleanessamento

Limpeza, tratamento de ausentes, encoding e normalização/padronização.

In [ ]:
# Imports
import pandas as pd
import numpy as np


In [ ]:
%load_ext autoreload
%autoreload 2
from utils import load_parquet_safe

df_path = "../data/df_model_raw.parquet"

df = load_parquet_safe(df_path, '01_eda.ipnyb')

print("Dados carregados com sucesso!")

In [ ]:
df.shape

In [ ]:
df

## Convertendo Sentinelas (Sentinelas -> NaNs)
O SINASC tem sentinelas que são ignorados pelo sistema, porém contentdo 9,99,999 como códigos a serem ignorados.

Objetivo: Equalizar os dados para que o Pandas consiga comparar as linhas de forma justa na etapa de duplicadas

### Como identificar as colunas que tem sentinela?

#### 1. Técnica da Frequência de Finais
Ao fazer um `value_counts().head(10)`conseguimos o Top 10 valores mais populares por coluna

In [ ]:
cols_to_check = df.columns

for col in cols_to_check:
    top_values = df[col].value_counts(normalize=True).head(10)
    print(f"Coluna: {col}")
    print(top_values)
    print("-" * 30)

### Como identificar o que é "lixo" do que é dado real? Será que todos os 9 na tabela toda são códigos de "ignorado"?

#### 2. Três critérios: Frequência, semântica e descontinuidade

Às vezes pelo nome da tabela essa informação consegue ser óbvia, mas outra vezes conseguimos seguir pelo o que os dados estão dizendo.

1. Onde o 9 e 99 são claramente SENTINELAS (lixo)
    - `MESPRENAT` **(99.0 com 2%)**: É sentinela. Ninguém começa o pré-natal no 99º mês. Note que o 9 aqui não aparece, o erro padrão do SINASC para meses é 99.
    - `ESTCIVMAE` e `ESCMAE2010` **(9.0 com ~0.2%)**: São sentinelas. Estão no final da lista, com frequências baixas, e o dicionário do SINASC reserva o 9 para "Ignorado" em variáveis categóricas codificadas de 1 a 5.
    - `SEXO` **(0 com 0.01%)**: O 0 aqui é sentinela para "Ignorado".

2. Onde o 9 é um VALOR REAL (dado válido)
    - Nestas colunas, o 9 faz parte de uma distribuição que vai diminuindo organicamente.
        - `QTDGESTANT, QTDPARTNOR, QTDFILVIVO`: * Observe a sequência: ...7, 8, 9, 10, 11...
            - O valor 9.0 aparece com uma frequência de **0.000650** (muito pequena). Se fosse um erro do sistema para "ignorado", esse número seria muito mais alto (como os 2% do `MESPRENAT`).
            - Aqui, 9 significa que a mãe realmente teve 9 gestações/partos. É um dado raro, mas biologicamente possível e importante para o modelo (multiparidade é fator de risco).
     
3. O caso especial: `RACACORMAE`
    - Note que em `RACACORMAE` apareceu um valor vazio com **2.5%**.
        - Isso é um "missing" disfarçado de string vazia. Você deve converter isso para NaN antes de qualquer outra coisa.


In [ ]:
# Cria uma cópia para limpeza
df_clean = df.copy()

# 1. Mapeamento cirúrgico de sentinelas
mapeamento_sentinelas = {
    'ESTCIVMAE': [9, 9.0],
    'ESCMAE2010': [9, 9.0],         # Mantemos o 0 (Sem escolaridade)
    'GRAVIDEZ': [9, 9.0],
    'SEXO': [0, 9, 9.0],            # No sexo, 0 é ignorado
    'MESPRENAT': [99, 99.0],
    'CONSPRENAT': [99, 99.0]
}

for col, sentinelas in mapeamento_sentinelas.items():
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace(sentinelas, np.nan)

# 2. Filtro de sanidade para IDADEPAI
# Removemos valores absurdos que distorcem a mediana
df_clean.loc[df_clean['IDADEPAI'] > 80, 'IDADEPAI'] = np.nan

# 3. Tratamento de strings vazias (Caso existam em colunas como RACACORMAE)
df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)


## Limpeza de Strings e Tipagem
Algumas strings na base que podem conter sugeira, como por exemplo `"MG"` tendo valores como `"MG "`(contendoespaços) o que é considerado diferente para o computador.

Objetivo: Remover espaços em branco e garantir que as colunas numéricas sejam de fato `float` ou `int`.

In [ ]:
# Tratamento de string vazia (ex: RACACORMAE)
df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)

## Identificação e Remoção de Duplicadas

Após a "normalização" dos dados (sentinelas viraram nulos e espaços foram removidos) conseguimos uma base melhor tratada para identificar duplicadas e removê-las.

Objetivo: Identificar e remover as duplicadas

In [ ]:
antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
depois = len(df_clean)

print(f"Registros removidos: {antes - depois}")

## Identificação e Tratamento de Nulos (imputação)

Agora que removemos as duplicadas precisamos tratar os nulos, isto é, nosso modelo não conseguiria cleanessar valores não numéricos.

In [ ]:
# 1. Ver contagem absoluta e percentual de nulos por coluna
null_summary = pd.DataFrame({
    'Nulos': df_clean.isnull().sum(),
    'Percentual (%)': (df_clean.isnull().mean() * 100).round(2)
})

# 2. Filtrar apenas as colunas que possuem algum nulo
print(null_summary[null_summary['Nulos'] > 0].sort_values(by='Nulos', ascending=False))

##### Aqui tens os passos finais — Preprocessing, por ordem de execução:

1. Imputação + Feature Engineering: Remover os nulos com medias/medianas.
   - **ps:** Por que não foi concluído nessa etapa? Acredito que a imputação pode envolver muito do feature Engineering, até porque podemos querer criar as features com base nos nulos.
   - Feature Engineering: Cria novas colunas de risco (ex: PN_TARDIO, HISTORICO_PERDA) usando os dados originais.

2. Encoding: Transforma colunas de texto (categóricas) em números usando pd.get_dummies (One-Hot) ou LabelEncoder.

3. Train/Test Split: Divide a base em Treino (80%) e Teste (20%). Importante: Usa stratify=y para manter a proporção de prematuros em ambos.

4. Scaling (Padronização): Usa o StandardScaler para colocar todas as variáveis na mesma escala (média 0, desvio padrão 1).